# Step 6: tolvaptan導入患者の病名抽出（導入日 ±30日のカルテ）

- 対象: `tolvaptan_study` で `ind_date` が入っている全員（181人の想定）
- カルテ: `ind_date` の前後30日（-30日〜+30日）
- 病名: カルテ内の `# 病名`（全角＃・半角#、♯、全角英数字に対応）。`#` から次の `#` または行末まで
- 分類: 心不全 / がん / 腎不全 / 肝硬変 / 低Na を 0/1
- 出力: ①患者別0/1表 ②根拠となった `#` 行の一覧（目視確認用）

In [12]:
import re
import sqlite3
import unicodedata
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

DB_PATH = Path("/Users/muna/Hana_research/data/db/Hana_Research.db")
OUTPUT_DIR = Path("/Users/muna/Hana_research/src/tolvaptan_study/")

FLAG_FILE = OUTPUT_DIR / "step6_disease_flags.csv"
EVIDENCE_FILE = OUTPUT_DIR / "step6_disease_evidence.csv"

WINDOW_DAYS = 30  # ind_date の前後それぞれ何日か

## 病名の辞書（ここを編集して探索）

検索対象は、全角→半角（NFKC）に正規化し、大文字にした文字列です。英字は大文字で書きます。

In [13]:
DISEASE_PATTERNS = {
    # 心不全
    "heart_failure": [
        r"心不全",
        r"(?<![A-Z0-9])(?:CHF|ADHF|HFPEF|HFREF|HFMREF|HF)(?![A-Z0-9])",
    ],
    # がん（腫瘍・リンパ腫・白血病を含む。良性腫瘍、ガンマは除く）
    "cancer": [
        r"癌",
        r"がん",
        r"ガン(?!マ)",
        r"悪性腫瘍",
        r"(?<!良性)腫瘍",
        r"リンパ腫",
        r"白血病",
    ],
    # 腎不全（CKD・慢性腎臓病・腎機能障害を含む）
    "renal_failure": [
        r"腎不全",
        r"(?<![A-Z0-9])CKD(?![A-Z0-9])",
        r"慢性腎臓病",
        r"腎機能障害",
    ],
    # 肝硬変
    "cirrhosis": [
        r"肝硬変",
    ],
    # 低Na（SIADHを含む）
    "hyponatremia": [
        r"低\s*NA(?![A-Z])",
        r"低ナトリウム",
        r"SIADH",
        r"HYPONATREMIA",
    ],
}

# がんは「既往・術後・寛解」を含めない（該当行は根拠CSVに counted=0 で残る）
CANCER_HISTORY_PATTERN = re.compile(
    r"既往|術後|手術後|切除後|寛解|治癒|根治|(?<![A-Z])S/P(?![A-Z])"
)

DISEASES = list(DISEASE_PATTERNS.keys())
DISEASE_REGEX = {d: re.compile("|".join(p)) for d, p in DISEASE_PATTERNS.items()}

## 1. 対象患者（tolvaptan_study で ind_date あり）

In [14]:
conn = sqlite3.connect(DB_PATH)

patients = pd.read_sql_query(
    '''
    SELECT Patient_ID, Study_ID, ind_date, date_precision
    FROM tolvaptan_study
    WHERE ind_date IS NOT NULL
    ''',
    conn,
)
patients["Patient_ID"] = pd.to_numeric(patients["Patient_ID"], errors="coerce").astype("Int64")
patients["ind_date"] = pd.to_datetime(patients["ind_date"], errors="coerce")

n_bad = patients["ind_date"].isna().sum()
print(f"対象患者数: {len(patients)}（ind_date を日付に直せない行: {n_bad}）")
print(patients["date_precision"].value_counts(dropna=False))
print("Study_ID が NULL:", patients["Study_ID"].isna().sum())
print("Patient_ID 重複:", patients["Patient_ID"].duplicated().sum())

対象患者数: 214（ind_date を日付に直せない行: 0）
date_precision
day    214
Name: count, dtype: int64
Study_ID が NULL: 0
Patient_ID 重複: 0


## 2. 対象患者のカルテを取得し、導入日の前後30日に絞る

`date_precision = month` の患者は ind_date を月初（1日）で補っているため、窓が実際の導入日からずれます。

In [15]:
patient_ids = [str(p) for p in patients["Patient_ID"].dropna().unique()]
placeholders = ",".join(["?"] * len(patient_ids))

karte = pd.read_sql_query(
    f'''
    SELECT Patient_ID, visit_datetime, karte_text
    FROM karte
    WHERE Patient_ID IN ({placeholders})
      AND karte_text IS NOT NULL
      AND karte_text != ''
    ''',
    conn,
    params=patient_ids,
)
conn.close()

karte["Patient_ID"] = pd.to_numeric(karte["Patient_ID"], errors="coerce").astype("Int64")
print(f"取得カルテ件数: {len(karte):,}")

ymd = karte["visit_datetime"].astype(str).str.extract(r"^(\d{4})[/-](\d{1,2})[/-](\d{1,2})")
karte["visit_date"] = pd.to_datetime(
    ymd[0] + "-" + ymd[1] + "-" + ymd[2], errors="coerce"
)
print("visit_date を読めなかったカルテ:", karte["visit_date"].isna().sum())

karte = karte.merge(
    patients[["Patient_ID", "Study_ID", "ind_date"]], on="Patient_ID", how="inner"
)
karte["days_from_ind"] = (karte["visit_date"] - karte["ind_date"]).dt.days

karte_window = karte[
    karte["days_from_ind"].between(-WINDOW_DAYS, WINDOW_DAYS)
].copy()
print(f"-{WINDOW_DAYS}日〜+{WINDOW_DAYS}日のカルテ: {len(karte_window):,} 件 "
      f"/ {karte_window['Patient_ID'].nunique()} 人")

取得カルテ件数: 26,951
visit_date を読めなかったカルテ: 0
-30日〜+30日のカルテ: 4,094 件 / 214 人


## 3. `#` で始まる病名を取り出す

全角＃、半角#、♯ を `#` にそろえ、全角英数字は半角にします。`# 〇〇` は、次の `#` または行末までを1つの病名欄として扱います。

In [16]:
HASH_SEG = re.compile(r"#([^#\r\n]*)")


def normalize(text):
    t = unicodedata.normalize("NFKC", str(text))
    t = t.replace("♯", "#").replace("＃", "#")
    return t.upper()


seg_rows = []
for rec in karte_window.itertuples(index=False):
    for line in re.split(r"[\r\n]+", normalize(rec.karte_text)):
        if "#" not in line:
            continue
        for seg in HASH_SEG.findall(line):
            seg = seg.strip()
            if seg:
                seg_rows.append({
                    "Study_ID": rec.Study_ID,
                    "Patient_ID": rec.Patient_ID,
                    "ind_date": rec.ind_date,
                    "visit_date": rec.visit_date,
                    "days_from_ind": rec.days_from_ind,
                    "segment": seg,
                    "line": line.strip(),
                })

segments = pd.DataFrame(seg_rows)
print(f"# の病名欄: {len(segments):,} 件 / {segments['Patient_ID'].nunique() if len(segments) else 0} 人")
segments.head(20)

# の病名欄: 11,227 件 / 214 人


,Study_ID,Patient_ID,ind_date,visit_date,days_from_ind,segment,line
0,P656628,160065,2016-03-31,2016-03-31,0,肝硬変(NASH疑い、CHILDPUGH C POINT10)、急性胆嚢炎既往,# 肝硬変(NASH疑い、CHILDPUGH C POINT10)、急性胆嚢炎既往
1,P656628,160065,2016-03-31,2016-03-31,0,認知症,# 認知症
2,P656628,160065,2016-03-31,2016-03-31,0,両側下腿蜂窩織炎、両側足背部潰瘍,# 両側下腿蜂窩織炎、両側足背部潰瘍
3,P656628,160065,2016-03-31,2016-03-31,0,難聴,# 難聴
4,P208041,150127,2016-03-07,2016-03-11,4,再発性直腸癌、腹膜播種、仙骨骨転移,# 再発性直腸癌、腹膜播種、仙骨骨転移
5,P208041,150127,2016-03-07,2016-03-11,4,左転子部発赤、仙骨部発赤,# 左転子部発赤、仙骨部発赤
6,P208041,150127,2016-03-07,2016-03-10,3,再発性直腸癌、腹膜播種、仙骨骨転移,# 再発性直腸癌、腹膜播種、仙骨骨転移
7,P208041,150127,2016-03-07,2016-03-10,3,左転子部発赤、仙骨部発赤,# 左転子部発赤、仙骨部発赤
8,P208041,150127,2016-03-07,2016-03-03,-4,再発性直腸癌、腹膜播種、仙骨骨転移,# 再発性直腸癌、腹膜播種、仙骨骨転移
9,P208041,150127,2016-03-07,2016-03-03,-4,左転子部発赤,# 左転子部発赤


## 4. 病名の判定（根拠行を作る）

In [17]:
evidence_parts = []
for disease, regex in DISEASE_REGEX.items():
    if segments.empty:
        break
    hit = segments[segments["segment"].str.contains(regex, na=False)].copy()
    hit["disease"] = disease
    hit["counted"] = 1
    hit["exclude_reason"] = ""
    if disease == "cancer":
        is_hist = hit["segment"].str.contains(CANCER_HISTORY_PATTERN, na=False)
        hit.loc[is_hist, "counted"] = 0
        hit.loc[is_hist, "exclude_reason"] = "既往・術後・寛解等"
    evidence_parts.append(hit)

evidence = (
    pd.concat(evidence_parts, ignore_index=True)
    if evidence_parts
    else pd.DataFrame(columns=list(segments.columns) + ["disease", "counted", "exclude_reason"])
)
evidence = evidence[[
    "Study_ID", "Patient_ID", "ind_date", "disease", "counted", "exclude_reason",
    "visit_date", "days_from_ind", "segment", "line",
]].sort_values(["Study_ID", "disease", "visit_date"]).reset_index(drop=True)

print(evidence.groupby(["disease", "counted"]).size().unstack(fill_value=0))

counted          0     1
disease                 
cancer         267   369
cirrhosis        0    89
heart_failure    0  1149
hyponatremia     0    72
renal_failure    0   346


## 5. 患者別の 0/1 表

In [18]:
flags = patients[["Study_ID", "Patient_ID", "ind_date", "date_precision"]].copy()

n_karte = karte_window.groupby("Patient_ID").size().rename("n_karte_in_window")
flags = flags.merge(n_karte, on="Patient_ID", how="left")
flags["n_karte_in_window"] = flags["n_karte_in_window"].fillna(0).astype(int)

if len(segments):
    n_seg = segments.groupby("Patient_ID").size().rename("n_hash_segments")
    flags = flags.merge(n_seg, on="Patient_ID", how="left")
flags["n_hash_segments"] = flags.get("n_hash_segments", 0)
flags["n_hash_segments"] = flags["n_hash_segments"].fillna(0).astype(int)

counted = evidence[evidence["counted"] == 1] if len(evidence) else evidence
for d in DISEASES:
    sub = counted[counted["disease"] == d] if len(counted) else counted
    ids = set(sub["Patient_ID"]) if len(sub) else set()
    flags[d] = flags["Patient_ID"].isin(ids).astype(int)

for d in DISEASES:
    sub = counted[counted["disease"] == d] if len(counted) else counted
    if len(sub):
        first = sub.groupby("Patient_ID")["visit_date"].min().rename(f"{d}_first_date")
        cnt = sub.groupby("Patient_ID").size().rename(f"{d}_hit_count")
        flags = flags.merge(first, on="Patient_ID", how="left").merge(cnt, on="Patient_ID", how="left")
    else:
        flags[f"{d}_first_date"] = pd.NaT
        flags[f"{d}_hit_count"] = 0
    flags[f"{d}_hit_count"] = flags[f"{d}_hit_count"].fillna(0).astype(int)

flags["disease_count"] = flags[DISEASES].sum(axis=1)

assert len(flags) == len(patients), "患者数が変わっています"
assert flags["Patient_ID"].is_unique, "Patient_ID が重複しています"
print(f"患者数: {len(flags)}")
flags.head(20)

患者数: 214


,Study_ID,Patient_ID,ind_date,date_precision,n_karte_in_window,n_hash_segments,heart_failure,cancer,renal_failure,cirrhosis,...,heart_failure_hit_count,cancer_first_date,cancer_hit_count,renal_failure_first_date,renal_failure_hit_count,cirrhosis_first_date,cirrhosis_hit_count,hyponatremia_first_date,hyponatremia_hit_count,disease_count
0,P044124,150004,2017-12-06,day,12,49,1,0,0,0,...,4,NaT,0,NaT,0,NaT,0,NaT,0,1
1,P432407,150009,2016-12-01,day,23,76,1,0,0,0,...,6,NaT,0,NaT,0,NaT,0,NaT,0,1
2,P725289,150027,2016-09-30,day,13,31,0,0,1,0,...,0,NaT,0,2016-09-06,3,NaT,0,NaT,0,1
3,P012835,150041,2016-05-10,day,30,68,1,0,1,0,...,9,NaT,0,2016-04-12,9,NaT,0,NaT,0,2
4,P661121,150061,2016-07-19,day,27,40,0,1,0,1,...,0,2016-06-20,12,NaT,0,2016-06-20,5,NaT,0,2
5,P277241,150067,2015-08-07,day,7,27,1,0,0,0,...,6,NaT,0,NaT,0,NaT,0,NaT,0,1
6,P602399,150068,2018-02-26,day,20,27,1,0,0,0,...,7,NaT,0,NaT,0,NaT,0,NaT,0,1
7,P561901,150091,2015-11-09,day,8,31,0,1,0,0,...,0,2015-10-19,3,NaT,0,NaT,0,2015-11-07,2,2
8,P396162,150105,2019-01-04,day,14,29,1,0,0,0,...,3,NaT,0,NaT,0,NaT,0,NaT,0,1
9,P241981,150119,2020-10-21,day,15,76,1,0,1,0,...,7,NaT,0,2020-10-06,5,NaT,0,NaT,0,2


## 6. 集計と確認

In [19]:
print(f"対象患者: {len(flags)}人")
print(f"窓内にカルテが1件もない患者: {(flags['n_karte_in_window'] == 0).sum()}人")
print(f"窓内にカルテはあるが # の病名欄がない患者: "
      f"{((flags['n_karte_in_window'] > 0) & (flags['n_hash_segments'] == 0)).sum()}人")
print()
print("病名別 該当人数")
for d in DISEASES:
    print(f"  {d:15s}: {flags[d].sum():4d} / {len(flags)}")
print()
print("5病名すべて0:", (flags["disease_count"] == 0).sum(), "人")
print(flags["disease_count"].value_counts().sort_index())

対象患者: 214人
窓内にカルテが1件もない患者: 0人
窓内にカルテはあるが # の病名欄がない患者: 0人

病名別 該当人数
  heart_failure  :  160 / 214
  cancer         :   48 / 214
  renal_failure  :   65 / 214
  cirrhosis      :   14 / 214
  hyponatremia   :   14 / 214

5病名すべて0: 7 人
disease_count
0      7
1    125
2     71
3     10
4      1
Name: count, dtype: int64


### 辞書の見直し用: どの病名にも当たらなかった `#` の病名欄（頻度順）

辞書の漏れ（「心機能低下」「透析」「HCC」など）を探すのに使います。

In [20]:
if len(segments):
    matched_any = pd.Series(False, index=segments.index)
    for d, regex in DISEASE_REGEX.items():
        matched_any |= segments["segment"].str.contains(regex, na=False)
    unmatched = segments.loc[~matched_any, "segment"].value_counts().head(60)
    display(unmatched.to_frame("件数"))

,件数
segment,
便秘症,226
便秘,149
2型糖尿病,109
脂質異常症,87
高尿酸血症,78
高血圧症,65
過活動膀胱,59
アルツハイマー型認知症,58
不眠症,57


### 根拠行の確認（病名ごとにランダム5件）

In [21]:
for d in DISEASES:
    sub = evidence[evidence["disease"] == d] if len(evidence) else evidence
    print(f"===== {d}（根拠 {len(sub)} 行 / 除外 {int((sub['counted'] == 0).sum()) if len(sub) else 0} 行）=====")
    if len(sub):
        display(sub.sample(min(5, len(sub)), random_state=0)[
            ["Study_ID", "visit_date", "days_from_ind", "counted", "segment"]
        ])

===== heart_failure（根拠 1149 行 / 除外 0 行）=====


,Study_ID,visit_date,days_from_ind,counted,segment
1941,P843981,2024-05-09,0,1,"慢性心不全、連合(AR,MR,TR)弁膜症"
2082,P902635,2025-08-24,-1,1,"慢性心不全HFPEF, 重度三尖弁閉鎖不全症&僧帽筋閉鎖不全症,慢性心房細動"
1889,P806043,2021-02-10,10,1,うっ血性心不全、心房頻拍(2:1)、発作性心房細動、軽度僧帽弁閉鎖不全症、高血圧症、徐脈
1901,P818832,2021-03-01,3,1,心不全・体液貯留
828,P276042,2021-04-22,-27,1,慢性心不全、心房細動、僧帽弁閉鎖不全症


===== cancer（根拠 636 行 / 除外 267 行）=====


,Study_ID,visit_date,days_from_ind,counted,segment
1675,P716446,2025-07-16,1,1,膵体部癌、腹膜播種、胸腹水貯留
1734,P756207,2021-01-07,28,1,左尿管癌、膀胱内転移、多発腹腔内リンパ節転移
1846,P797180,2025-12-03,14,0,前立腺癌術後(2004)
1947,P847143,2023-03-30,-19,1,腎腫瘍疑い(2022.5月);良性疑い
287,P104181,2023-06-12,6,1,膵尾部癌、多発肝転移、胃脾浸潤 STAGEIV


===== renal_failure（根拠 346 行 / 除外 0 行）=====


,Study_ID,visit_date,days_from_ind,counted,segment
2140,P911533,2019-02-26,21,1,慢性腎臓病、高血圧症
348,P132534,2025-07-24,14,1,"持続性心房細動、慢性心不全HFPEF, 高血圧症、慢性腎臓病"
1964,P847143,2023-04-23,5,1,慢性腎臓病、高尿酸血症
1322,P540030,2021-10-21,10,1,"慢性心不全、重度大動脈弁狭窄症(LOW GRADIENT AS S/O), 重度僧帽弁閉鎖不全症、持続性心房細動、慢性腎臓病、睡眠時無呼吸症候群"
31,P012835,2016-05-24,14,1,慢性腎臓病


===== cirrhosis（根拠 89 行 / 除外 0 行）=====


,Study_ID,visit_date,days_from_ind,counted,segment
129,P060581,2024-11-06,-1,1,肝細胞癌、腹膜浸潤、肝硬変(アルコール性)
253,P101811,2025-06-05,4,1,"肝細胞癌、アルコール性肝硬変(CHILD-PUGH C), 下大静脈浸潤、門脈腫瘍塞栓、肺転移"
1478,P624776,2022-08-03,-2,1,アルコール性肝硬変、腹水貯留
973,P340079,2021-06-23,14,1,アルコール性肝硬変、肝細胞癌
1618,P661121,2016-06-20,-29,1,肝細胞癌、C型肝硬変(CHILD-PUGH A)、肝細胞ガン破裂後


===== hyponatremia（根拠 72 行 / 除外 0 行）=====


,Study_ID,visit_date,days_from_ind,counted,segment
1017,P368183,2021-09-01,0,1,"低ナトリウム血症既往(原因不明,2020/5,6)2021年4月 138"
1018,P368183,2021-09-08,7,1,"低ナトリウム血症既往(原因不明,2020/5,6)2021年4月 138"
1338,P561901,2015-11-07,-2,1,下腿浮腫、低ナトリウム血症
806,P269000,2023-09-07,3,1,低ナトリウム血症
1021,P368183,2021-09-29,28,1,"低ナトリウム血症既往(原因不明,2020/5,6)2021年4月 138"


## 7. CSV に保存

In [22]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

flags.to_csv(FLAG_FILE, index=False, encoding="utf-8-sig")
evidence.to_csv(EVIDENCE_FILE, index=False, encoding="utf-8-sig")

print("患者別 0/1 表 :", FLAG_FILE, f"({len(flags)}行)")
print("根拠の一覧   :", EVIDENCE_FILE, f"({len(evidence)}行)")

患者別 0/1 表 : /Users/muna/Hana_research/src/tolvaptan_study/step6_disease_flags.csv (214行)
根拠の一覧   : /Users/muna/Hana_research/src/tolvaptan_study/step6_disease_evidence.csv (2292行)
